# RNN 重要改进详解

## RNN 的演进路线

```
RNN (1986) -> LSTM (1997) -> GRU (2014) -> Bidirectional RNN -> Seq2Seq + Attention (2014)
    -> Transformer (2017, 替代 RNN 的主流)
```

本课深入讲解:
1. LSTM 门控机制深入
2. GRU 简化设计
3. 双向 RNN
4. Seq2Seq 编码器-解码器
5. 注意力机制 (RNN 版本)
6. RNN 训练技巧

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. LSTM 门控机制深入

LSTM 通过三个门控制信息流动:

```
遗忘门 f: 决定丢弃哪些旧信息 (C_{t-1} 中哪些要忘掉)
输入门 i: 决定存入哪些新信息 (哪些新信息写入 C_t)
输出门 o: 决定输出哪些信息 (C_t 中哪些作为 h_t 输出)
```

关键: 细胞状态 C_t 通过加法更新 (而非乘法), 梯度可以直接流过!

In [ ]:
# 手动实现 LSTM，逐步展示每个门的作用
class ManualLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # 将四个门的权重合并为一个大矩阵，更高效
        self.W_ih = nn.Linear(input_size, 4 * hidden_size)
        self.W_hh = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(self, x, init_states=None):
        batch_size, seq_len, _ = x.shape
        h, c = (torch.zeros(batch_size, self.hidden_size, device=x.device),
                torch.zeros(batch_size, self.hidden_size, device=x.device))
        if init_states is not None:
            h, c = init_states

        outputs = []
        gates_history = []

        for t in range(seq_len):
            # 合并计算所有门
            gates = self.W_ih(x[:, t]) + self.W_hh(h)
            i, f, g, o = gates.chunk(4, dim=1)  # 分成4份

            i = torch.sigmoid(i)  # 输入门
            f = torch.sigmoid(f)  # 遗忘门
            g = torch.tanh(g)     # 候选值
            o = torch.sigmoid(o)  # 输出门

            # 更新细胞状态 (加法! 不是乘法!)
            c = f * c + i * g
            # 输出隐藏状态
            h = o * torch.tanh(c)

            outputs.append(h)
            gates_history.append({'i': i, 'f': f, 'g': g, 'o': o, 'c': c, 'h': h})

        outputs = torch.stack(outputs, dim=1)
        return outputs, (h, c), gates_history

lstm = ManualLSTM(input_size=8, hidden_size=16)
x = torch.randn(1, 10, 8)
out, (hn, cn), gates = lstm(x)
print(f"LSTM 输出: {out.shape}, h_n: {hn.shape}, c_n: {cn.shape}")

In [ ]:
# 可视化 LSTM 门控值随时间变化
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
gate_names = ['i (输入门)', 'f (遗忘门)', 'o (输出门)', 'c (细胞状态)']
gate_keys = ['i', 'f', 'o', 'c']

for ax, name, key in zip(axes.flatten(), gate_names, gate_keys):
    data = torch.stack([g[key][0] for g in gates]).detach().numpy()
    im = ax.imshow(data.T, aspect='auto', cmap='RdBu_r')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Hidden Dim')
    ax.set_title(name)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('LSTM Gate Values Over Time', fontsize=14)
plt.tight_layout()
plt.show()

print("输入门 i: 控制新信息写入 (0=不写入, 1=全部写入)")
print("遗忘门 f: 控制旧信息保留 (0=全部遗忘, 1=全部保留)")
print("输出门 o: 控制信息输出 (0=不输出, 1=全部输出)")
print("细胞状态 c: 长期记忆, 通过加法更新, 梯度可直接流过")

In [ ]:
# LSTM vs RNN 梯度流对比
print("=== 为什么 LSTM 能解决梯度消失? ===")
print()
print("RNN 的递推:")
print("  h_t = tanh(W_hh * h_{t-1} + W_ih * x_t + b)")
print("  梯度: dh_t/dh_{t-1} = diag(tanh') * W_hh")
print("  经过 T 步: (diag(tanh') * W_hh)^T")
print("  tanh' 最大 1, W_hh 特征值 < 1 时梯度消失")
print()
print("LSTM 的细胞状态递推:")
print("  C_t = f_t * C_{t-1} + i_t * g_t")
print("  梯度: dC_t/dC_{t-1} = f_t  (逐元素乘法)")
print("  经过 T 步: f_T * f_{T-1} * ... * f_1")
print("  遗忘门 f_t 接近 1 时, 梯度几乎不衰减!")
print()
print("关键区别: LSTM 的 C_t 更新是加法 (f*C + i*g),")
print("而 RNN 的 h_t 更新是全替换 (tanh(W*h + W*x))")
print("加法更新让梯度有"高速公路"可以直接流过")

---
## 2. GRU: LSTM 的简化版

GRU 合并了细胞状态和隐藏状态, 只有 2 个门:

```
LSTM: 3个门 (遗忘门f, 输入门i, 输出门o) + 细胞状态C
GRU:  2个门 (重置门r, 更新门z) + 无独立细胞状态
```

GRU 公式:
```
z_t = sigmoid(W_z * [h_{t-1}, x_t])   # 更新门
r_t = sigmoid(W_r * [h_{t-1}, x_t])   # 重置门
h_tilde = tanh(W * [r_t * h_{t-1}, x_t])  # 候选值
h_t = (1 - z_t) * h_{t-1} + z_t * h_tilde # 更新
```

In [ ]:
# 手动实现 GRU
class ManualGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ih = nn.Linear(input_size, 3 * hidden_size)
        self.W_hh = nn.Linear(hidden_size, 3 * hidden_size)

    def forward(self, x, h=None):
        batch_size, seq_len, _ = x.shape
        if h is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        outputs = []
        for t in range(seq_len):
            gates = self.W_ih(x[:, t]) + self.W_hh(h)
            r, z, n = gates.chunk(3, dim=1)

            r = torch.sigmoid(r)  # 重置门
            z = torch.sigmoid(z)  # 更新门
            n = torch.tanh(n)     # 候选值 (注意: 实际实现中 n 的计算更复杂)

            h = (1 - z) * h + z * n  # 插值更新
            outputs.append(h)

        return torch.stack(outputs, dim=1), h

gru = ManualGRU(input_size=8, hidden_size=16)
x = torch.randn(1, 10, 8)
out, hn = gru(x)
print(f"GRU 输出: {out.shape}, h_n: {hn.shape}")
print()
print("GRU vs LSTM:")
print("  GRU 参数更少 (2个门 vs 4个门)")
print("  GRU 没有独立的细胞状态")
print("  GRU 的更新门 z 同时承担了遗忘和输入的功能")
print("  效果通常相当, GRU 在小数据集上可能更好")

In [ ]:
# LSTM vs GRU 参数量对比
input_size = 100
hidden_size = 128

lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
gru = nn.GRU(input_size, hidden_size, batch_first=True)
rnn = nn.RNN(input_size, hidden_size, batch_first=True)

print(f"RNN  参数量: {sum(p.numel() for p in rnn.parameters()):>8,}")
print(f"LSTM 参数量: {sum(p.numel() for p in lstm.parameters()):>8,} ({sum(p.numel() for p in lstm.parameters())/sum(p.numel() for p in rnn.parameters()):.1f}x RNN)")
print(f"GRU  参数量: {sum(p.numel() for p in gru.parameters()):>8,} ({sum(p.numel() for p in gru.parameters())/sum(p.numel() for p in rnn.parameters()):.1f}x RNN)")
print()
print("参数量公式:")
print(f"  RNN:  hidden_size * (input_size + hidden_size + 1) = {hidden_size*(input_size+hidden_size+1):,}")
print(f"  LSTM: 4 * hidden_size * (input_size + hidden_size + 1) = {4*hidden_size*(input_size+hidden_size+1):,}")
print(f"  GRU:  3 * hidden_size * (input_size + hidden_size + 1) = {3*hidden_size*(input_size+hidden_size+1):,}")

---
## 3. 双向 RNN

标准 RNN 只看过去, 双向 RNN 同时看过去和未来:

```
前向: h1 -> h2 -> h3 -> h4  (看过去)
后向: h4 -> h3 -> h2 -> h1  (看未来)
输出: concat(h_forward, h_backward)
```

适用于: 文本分类、命名实体识别等需要完整上下文的任务
不适用于: 实时生成 (未来信息不可用)

In [ ]:
# 双向 LSTM
bilstm = nn.LSTM(input_size=32, hidden_size=64, num_layers=2,
                  batch_first=True, bidirectional=True, dropout=0.2)

x = torch.randn(4, 20, 32)
output, (hn, cn) = bilstm(x)

print(f"输入: {x.shape}")
print(f"输出: {output.shape}")  # (4, 20, 128) = hidden_size * 2
print(f"h_n: {hn.shape}")        # (4, 4, 64) = num_layers * 2
print()
print("双向 LSTM 输出维度 = hidden_size * 2")
print("前向和后向的隐藏状态在最后一个维度拼接")
print()
print("提取前向和后向的最后隐藏状态:")
h_forward = hn[-2]   # 前向最后一层
h_backward = hn[-1]  # 后向最后一层
h_concat = torch.cat([h_forward, h_backward], dim=1)
print(f"  h_forward: {h_forward.shape}")
print(f"  h_backward: {h_backward.shape}")
print(f"  拼接后: {h_concat.shape}")

In [ ]:
# 双向 vs 单向 对比实验
torch.manual_seed(42)
n_samples = 500
seq_len = 30
input_size = 16

X = torch.randn(n_samples, seq_len, input_size)
y = torch.randint(0, 3, (n_samples,))

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

class BiRNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, bidirectional=True):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2,
                           batch_first=True, bidirectional=bidirectional, dropout=0.2)
        direction = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_size * direction, num_classes)

    def forward(self, x):
        output, _ = self.lstm(x)
        out = output[:, -1, :]  # 取最后时间步
        return self.fc(out)

def train_classifier(model, epochs=30):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    accs = []
    for epoch in range(epochs):
        correct, total = 0, 0
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            correct += (logits.argmax(1) == y_batch).sum().item()
            total += X_batch.size(0)
        accs.append(correct / total)
    return accs

torch.manual_seed(42)
uni_accs = train_classifier(BiRNNClassifier(16, 32, 3, bidirectional=False))
torch.manual_seed(42)
bi_accs = train_classifier(BiRNNClassifier(16, 32, 3, bidirectional=True))

plt.figure(figsize=(8, 5))
plt.plot(uni_accs, label='Unidirectional LSTM', color='red')
plt.plot(bi_accs, label='Bidirectional LSTM', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Unidirectional vs Bidirectional LSTM')
plt.legend()
plt.grid(True)
plt.show()

print(f"单向 LSTM 最终准确率: {uni_accs[-1]:.4f}")
print(f"双向 LSTM 最终准确率: {bi_accs[-1]:.4f}")

---
## 4. Seq2Seq: 编码器-解码器

Seq2Seq 用于序列到序列的转换 (翻译、摘要、对话等):

```
编码器: 输入序列 -> 上下文向量 (最后隐藏状态)
解码器: 上下文向量 -> 输出序列
```

问题: 固定长度的上下文向量是信息瓶颈。

In [ ]:
# Seq2Seq 基础实现
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (h, c) = self.lstm(embedded)
        return output, (h, c)

class Decoder(nn.Module):
    def __init__(self, output_size, hidden_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.lstm(embedded, hidden)
        prediction = self.fc(output)
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size)
        _, hidden = self.encoder(src)

        # 解码器第一个输入是目标序列的起始符
        input_token = trg[:, 0:1]

        for t in range(trg_len):
            prediction, hidden = self.decoder(input_token, hidden)
            outputs[:, t:t+1] = prediction

            # Teacher Forcing: 以一定概率用真实目标作为下一步输入
            if np.random.random() < teacher_forcing_ratio:
                input_token = trg[:, t:t+1]
            else:
                input_token = prediction.argmax(2)

        return outputs

vocab_size = 100
hidden_size = 64
encoder = Encoder(vocab_size, hidden_size)
decoder = Decoder(vocab_size, hidden_size)
seq2seq = Seq2Seq(encoder, decoder)

src = torch.randint(0, vocab_size, (4, 10))
trg = torch.randint(0, vocab_size, (4, 8))
out = seq2seq(src, trg)
print(f"源序列: {src.shape}, 目标序列: {trg.shape}")
print(f"输出: {out.shape}")
print(f"参数量: {sum(p.numel() for p in seq2seq.parameters()):,}")

In [ ]:
# Teacher Forcing 解释
print("=== Teacher Forcing ===")
print()
print("训练解码器时, 每一步的输入有两种选择:")
print()
print("1. 用模型自己的预测 (自回归):")
print("   优点: 推理一致, 更稳定")
print("   缺点: 早期训练时预测很差, 误差累积")
print()
print("2. 用真实的目标 (Teacher Forcing):")
print("   优点: 训练更快, 每步都从正确输入开始")
print("   缺点: 推理时没有真实目标, 存在偏差")
print()
print("实际做法: 以概率 p 使用 Teacher Forcing")
print("  训练初期: p=1.0 (全用真实目标)")
print("  训练后期: p=0.5 (混合使用)")
print("  Scheduled Sampling: p 逐渐减小")

---
## 5. 注意力机制 (RNN 版本)

Seq2Seq 的瓶颈: 编码器的所有信息压缩到一个固定长度的向量。

注意力机制: 解码器每一步都"回头看"编码器的所有输出, 动态选择相关信息。

```
1. 计算注意力分数: score = decoder_hidden @ encoder_outputs^T
2. 归一化: attn_weights = softmax(score)
3. 加权求和: context = attn_weights @ encoder_outputs
4. 拼接: input = concat(context, decoder_hidden)
```

In [ ]:
# Bahdanau 注意力 (加性注意力)
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W1 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W2 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (B, H)
        # encoder_outputs: (B, S, H)
        score = self.v(torch.tanh(self.W1(encoder_outputs) + self.W2(decoder_hidden.unsqueeze(1))))
        attn_weights = torch.softmax(score, dim=1)  # (B, S, 1)
        context = (attn_weights * encoder_outputs).sum(dim=1)  # (B, H)
        return context, attn_weights.squeeze(-1)

# Luong 注意力 (点积注意力, 更简单)
class LuongAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (B, H)
        # encoder_outputs: (B, S, H)
        score = decoder_hidden @ encoder_outputs.transpose(1, 2)  # (B, 1, S)
        attn_weights = torch.softmax(score, dim=-1)  # (B, 1, S)
        context = attn_weights @ encoder_outputs  # (B, 1, H)
        context = context.squeeze(1)  # (B, H)
        return context, attn_weights.squeeze(1)

attn = LuongAttention(64)
dec_h = torch.randn(2, 64)
enc_out = torch.randn(2, 10, 64)
context, weights = attn(dec_h, enc_out)
print(f"解码器隐藏状态: {dec_h.shape}")
print(f"编码器输出: {enc_out.shape}")
print(f"上下文向量: {context.shape}")
print(f"注意力权重: {weights.shape}")

In [ ]:
# 带注意力的 Seq2Seq
class AttnDecoder(nn.Module):
    def __init__(self, output_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(2 * hidden_size, hidden_size, batch_first=True)
        self.attention = LuongAttention(hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, encoder_outputs):
        embedded = self.embedding(x)
        h = hidden[0].squeeze(0)
        context, attn_weights = self.attention(h, encoder_outputs)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, hidden = self.lstm(rnn_input, hidden)
        prediction = self.fc(output)
        return prediction, hidden, attn_weights

class AttnSeq2Seq(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.encoder = Encoder(vocab_size, hidden_size)
        self.decoder = AttnDecoder(vocab_size, hidden_size)

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, trg_len, vocab_size)

        encoder_outputs, hidden = self.encoder(src)
        input_token = trg[:, 0:1]

        for t in range(trg_len):
            prediction, hidden, _ = self.decoder(input_token, hidden, encoder_outputs)
            outputs[:, t:t+1] = prediction
            input_token = trg[:, t:t+1] if np.random.random() < teacher_forcing_ratio else prediction.argmax(2)

        return outputs

model = AttnSeq2Seq(100, 64)
src = torch.randint(0, 100, (4, 10))
trg = torch.randint(0, 100, (4, 8))
out = model(src, trg)
print(f"带注意力的 Seq2Seq: {src.shape} -> {out.shape}")
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 注意力权重可视化
attn_vis = LuongAttention(64)
dec_h = torch.randn(1, 64)
enc_out = torch.randn(1, 8, 64)

# 模拟解码过程中每一步的注意力
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for t in range(8):
    dec_h_t = torch.randn(1, 64)
    context, weights = attn_vis(dec_h_t, enc_out)
    ax = axes[t // 4, t % 4]
    ax.bar(range(8), weights[0].detach().numpy())
    ax.set_title(f'Decoder Step {t+1}')
    ax.set_xlabel('Source Position')
    ax.set_ylabel('Attention')
    ax.set_ylim(0, 1)

plt.suptitle('Attention Weights at Each Decoder Step', fontsize=14)
plt.tight_layout()
plt.show()

print("每个解码步关注源序列的不同位置")
print("注意力机制解决了固定长度上下文向量的瓶颈问题")

---
## 6. RNN 训练技巧

In [ ]:
print("=== RNN 训练最佳实践 ===")
print()
print("1. 梯度裁剪 (必须!)")
print("   torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("   在 loss.backward() 之后, optimizer.step() 之前")
print()
print("2. 使用 LSTM 或 GRU (不要用普通 RNN)")
print("   除非有特殊原因, 默认用 LSTM 或 GRU")
print()
print("3. batch_first=True")
print("   输入形状 (batch, seq, feature) 更直观")
print()
print("4. 合适的序列长度")
print("   RNN 处理长序列困难, 考虑截断或分层")
print("   超过 200-300 步建议用 Transformer")
print()
print("5. 堆叠层数")
print("   通常 2-3 层足够, 太深容易过拟合")
print()
print("6. Dropout 位置")
print("   层间 dropout (nn.LSTM 的 dropout 参数)")
print("   不要在循环内部加 dropout")
print()
print("7. 隐藏状态初始化")
print("   默认全零即可")
print("   也可以学习初始隐藏状态 (nn.Parameter)")

In [ ]:
# 可学习初始隐藏状态
class LSTMWithLearnableInit(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.h0 = nn.Parameter(torch.randn(num_layers, 1, hidden_size) * 0.01)
        self.c0 = nn.Parameter(torch.randn(num_layers, 1, hidden_size) * 0.01)

    def forward(self, x):
        batch_size = x.shape[0]
        h0 = self.h0.expand(-1, batch_size, -1)
        c0 = self.c0.expand(-1, batch_size, -1)
        output, _ = self.lstm(x, (h0, c0))
        return output

model = LSTMWithLearnableInit(32, 64)
x = torch.randn(4, 10, 32)
print(f"输出: {model(x).shape}")
print("可学习的初始隐藏状态, 让模型自己决定最优起点")

In [ ]:
# RNN vs Transformer 选择指南
print("=== RNN vs Transformer 选择指南 ===")
print()
print("用 RNN (LSTM/GRU) 的场景:")
print("  - 小数据集 (< 10万样本)")
print("  - 短序列 (< 200步)")
print("  - 实时/流式处理 (必须逐步处理)")
print("  - 资源受限 (嵌入式设备)")
print("  - 时间序列预测")
print()
print("用 Transformer 的场景:")
print("  - 大数据集 (> 10万样本)")
print("  - 长序列 (需要并行训练)")
print("  - NLP 任务 (文本分类、生成、翻译)")
print("  - 有充足 GPU 资源")
print()
print("混合方案:")
print("  - Conformer: CNN + Transformer (语音识别)")
print("  - LSTM + Attention: RNN 加注意力")
print("  - 预训练 Transformer + 微调 LSTM (轻量化部署)")

---
## 总结

### RNN 改进路线

| 改进 | 解决的问题 | 核心机制 |
|------|-----------|----------|
| LSTM | 梯度消失 | 门控 + 细胞状态(加法更新) |
| GRU | LSTM 参数多 | 合并门控, 2门替代4门 |
| 双向 RNN | 只看过去 | 前向+后向同时处理 |
| Seq2Seq | 序列转换 | 编码器+解码器 |
| 注意力 | 固定长度瓶颈 | 动态关注编码器输出 |
| Transformer | RNN 串行瓶颈 | 自注意力并行计算 |

### 关键公式

```
LSTM: C_t = f_t * C_{t-1} + i_t * g_t  (加法更新!)
GRU:  h_t = (1-z_t) * h_{t-1} + z_t * h_tilde
Attention: context = softmax(score) @ encoder_outputs
```

### 实践建议
- 默认用 LSTM 或 GRU, 不要用普通 RNN
- 训练时必须梯度裁剪
- 短序列用 RNN, 长序列/大数据用 Transformer
- 双向 RNN 适用于离线任务
- 注意力机制是 RNN 到 Transformer 的桥梁